# HR Employee Attrition — Tableau Dashboard
**Author:** Shantanu Deshmukh &nbsp;|&nbsp; **Data Prep:** Python · Pandas &nbsp;|&nbsp; **Dashboard:** Tableau Public

---

## Purpose
This notebook generates and cleans an HR Employee Attrition dataset, performs exploratory analysis, and exports a Tableau-ready CSV.

After running this notebook, open `TABLEAU_GUIDE.md` for step-by-step instructions to build your live Tableau Public dashboard.

## Dataset
Synthetic dataset modeled on the IBM HR Analytics Employee Attrition dataset.
- **1,470 employees** across departments, job roles, and demographics
- Features: age, department, income, tenure, satisfaction scores, overtime, attrition status

## Business Questions (answered in the Tableau dashboard)
1. What is the overall attrition rate?
2. Which departments and job roles have the highest attrition?
3. Does age group predict attrition?
4. How does monthly income relate to employee retention?
5. What is the impact of overtime on attrition?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 8)
plt.rcParams['font.family']    = 'sans-serif'
np.random.seed(99)
N = 1470
print("✓ Libraries loaded. Generating HR dataset...")

In [ ]:
# ── GENERATE SYNTHETIC HR DATASET ────────────────────────────
dept   = np.random.choice(['Sales','Research & Development','Human Resources'],
                           N, p=[0.32,0.61,0.07])

role_map = {
    'Sales':                  ['Sales Executive','Sales Representative','Manager'],
    'Research & Development': ['Research Scientist','Laboratory Technician',
                               'Manufacturing Director','Research Director','Healthcare Representative'],
    'Human Resources':        ['Human Resources','Manager']
}
role = [np.random.choice(role_map[d]) for d in dept]

age           = np.random.normal(37,9,N).clip(18,60).astype(int)
yrs_company   = np.random.exponential(7,N).clip(0,40).astype(int)
yrs_role      = np.minimum(yrs_company, np.random.exponential(4,N).clip(0,20).astype(int))
job_level     = np.random.choice([1,2,3,4,5], N, p=[0.26,0.36,0.22,0.12,0.04])
base          = {1:2500,2:4500,3:7000,4:11000,5:17000}
monthly_inc   = np.array([int(base[l]+np.random.normal(0,base[l]*0.2)) for l in job_level]).clip(1000,20000)
education     = np.random.choice(['Below College','College','Bachelor','Master','Doctor'],
                                  N, p=[0.12,0.19,0.39,0.27,0.03])
gender        = np.random.choice(['Male','Female'], N, p=[0.60,0.40])
marital       = np.random.choice(['Single','Married','Divorced'], N, p=[0.32,0.46,0.22])
overtime      = np.random.choice(['Yes','No'], N, p=[0.28,0.72])
job_sat       = np.random.choice([1,2,3,4], N, p=[0.20,0.20,0.30,0.30])
wlb           = np.random.choice([1,2,3,4], N, p=[0.10,0.23,0.38,0.29])
env_sat       = np.random.choice([1,2,3,4], N, p=[0.19,0.19,0.31,0.31])
dist_home     = np.random.exponential(9,N).clip(1,29).astype(int)
training      = np.random.choice(range(7), N, p=[0.06,0.10,0.31,0.28,0.15,0.07,0.03])
num_companies = np.random.choice(range(10), N, p=[0.10,0.26,0.20,0.17,0.10,0.06,0.05,0.03,0.02,0.01])

attr_p = (0.05
          + 0.15*(overtime=='Yes')
          + 0.10*(job_sat<=2)
          + 0.08*(wlb==1)
          + 0.07*(yrs_company<=2)
          + 0.06*(job_level==1)
          + 0.05*(marital=='Single')
          + 0.04*(dist_home>20)
          - 0.003*monthly_inc/1000
          + np.random.normal(0,0.04,N)).clip(0.01,0.90)

attrition = np.where(np.random.uniform(0,1,N) < attr_p, 'Yes','No')

df = pd.DataFrame({
    'EmployeeID':np.arange(1,N+1),'Age':age,'Department':dept,'JobRole':role,
    'Gender':gender,'MaritalStatus':marital,'Education':education,
    'JobLevel':job_level,'MonthlyIncome':monthly_inc,
    'YearsAtCompany':yrs_company,'YearsInCurrentRole':yrs_role,
    'NumCompaniesWorked':num_companies,'DistanceFromHome':dist_home,
    'OverTime':overtime,'JobSatisfaction':job_sat,
    'WorkLifeBalance':wlb,'EnvironmentSatisfaction':env_sat,
    'TrainingTimesLastYear':training,'Attrition':attrition
})

print(f"✓ Dataset generated: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Attrition Rate: {(df['Attrition']=='Yes').mean()*100:.1f}%")
df.head()

In [ ]:
# ── EXPLORATORY ANALYSIS ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ATTR   = '#c0392b'
RETAIN = '#2e6da4'

# 1 — Attrition by Department
da = (df.groupby('Department')['Attrition']
       .apply(lambda x: (x=='Yes').mean()*100).reset_index())
da.columns = ['Department','Rate']
axes[0,0].bar(da['Department'], da['Rate'],
              color=[RETAIN, ATTR, '#e67e22'][:len(da)], edgecolor='white')
axes[0,0].set_title('Attrition Rate by Department', fontweight='bold')
axes[0,0].set_ylabel('Attrition Rate (%)')
axes[0,0].tick_params(axis='x', rotation=10)
for i,(_, row) in enumerate(da.iterrows()):
    axes[0,0].text(i, row['Rate']+.2, f"{row['Rate']:.1f}%", ha='center', fontweight='bold')

# 2 — Attrition by Age Group
df['AgeGroup'] = pd.cut(df['Age'], bins=[18,25,35,45,60],
                         labels=['18–25','26–35','36–45','46–60'])
aa = df.groupby('AgeGroup', observed=False)['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
axes[0,1].plot(aa.index, aa.values, 'o-', color=ATTR, lw=2.5, ms=10)
axes[0,1].fill_between(range(len(aa)), aa.values, alpha=.12, color=ATTR)
axes[0,1].set_title('Attrition Rate by Age Group', fontweight='bold')
axes[0,1].set_ylabel('Attrition Rate (%)')

# 3 — Income by Attrition
df.boxplot(column='MonthlyIncome', by='Attrition', ax=axes[1,0],
           boxprops=dict(color='#1a3a5c'), medianprops=dict(color='red',linewidth=2))
axes[1,0].set_title('Monthly Income by Attrition Status')
axes[1,0].set_xlabel('Attrition')
axes[1,0].set_ylabel('Monthly Income ($)')
plt.sca(axes[1,0]); plt.title('Monthly Income by Attrition Status')

# 4 — Overtime impact
ot = df.groupby('OverTime')['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
bars = axes[1,1].bar(ot.index, ot.values, color=[RETAIN,ATTR], edgecolor='white')
axes[1,1].set_title('Attrition Rate: Overtime vs. No Overtime', fontweight='bold')
axes[1,1].set_ylabel('Attrition Rate (%)')
for b, v in zip(bars, ot.values):
    axes[1,1].text(b.get_x()+b.get_width()/2, v+.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('HR Attrition — Exploratory Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(r'project2-hr-tableau-dashboard','hr_eda.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EXPORT CLEAN CSV FOR TABLEAU ─────────────────────────────
df['AttritionBinary'] = (df['Attrition']=='Yes').astype(int)
df['IncomeLevel']     = pd.cut(df['MonthlyIncome'],
    bins=[0,3000,6000,10000,21000],
    labels=['Low (<$3k)','Mid ($3–6k)','High ($6–10k)','Very High (>$10k)'])
df['TenureGroup']     = pd.cut(df['YearsAtCompany'],
    bins=[-1,2,5,10,20,40],
    labels=['0–2 yrs','3–5 yrs','6–10 yrs','11–20 yrs','20+ yrs'])
df['SatisfactionLabel'] = df['JobSatisfaction'].map({1:'Low',2:'Medium',3:'High',4:'Very High'})
df['AgeGroup']          = pd.cut(df['Age'], bins=[18,25,35,45,60],
                                  labels=['18–25','26–35','36–45','46–60'])

out_path = os.path.join(r'project2-hr-tableau-dashboard','hr_attrition_clean.csv')
df.to_csv(out_path, index=False)
print(f"✓ Exported: hr_attrition_clean.csv")
print(f"  Rows: {len(df):,} | Columns: {df.shape[1]}")
print()
print("NEXT STEPS:")
print("  1. Open Tableau Public (free at public.tableau.com)")
print("  2. Connect to hr_attrition_clean.csv")
print("  3. Follow TABLEAU_GUIDE.md to build your dashboard")
print("  4. Publish and add the link to your resume!")